In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [2]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [3]:
## Q1
from embedder import Embedder

query = "How does approximate nearest neighbor search work?"

embedder = Embedder()
query_vect = embedder.encode(query)
query_vect[0]

2026-06-29 12:53:54.643312789 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


np.float64(-0.02058203437252893)

In [ ]:
## Q2
filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

filter_doc = [doc for doc in documents if doc["filename"] == filename][0]
query_1 = filter_doc["content"]
query_vect_1 = embedder.encode(query_1)

query_vect_1.dot(query_vect)


np.float64(0.36107026789538205)

In [5]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [11]:
chunks_text = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(chunks_text)

In [ ]:
## Q3
import numpy as np

scores = X.dot(query_vect)
idx = np.argmax(scores)
chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [24]:
# Extra validation
"approximate" in chunks[idx]["content"]

True

In [29]:
## Q4
from minsearch import VectorSearch

query_q4 = "What metric do we use to evaluate a search engine?"
query_vect_q4 = embedder.encode(query_q4)

vindex = VectorSearch()
vindex.fit(X, chunks)
vindex.search(query_vect_q4, num_results=1)

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

In [41]:
## Q5
from minsearch import Index

index = Index(
    text_fields=["content"],
)
index.fit(chunks)

query_q5 = "How do I store vectors in PostgreSQL?"
query_vect_q5 = embedder.encode(query_q5)

result_text = index.search(query_q5, num_results=5)
result_vector = vindex.search(query_vect_q5, num_results=5)

vect_filenames = {res["filename"] for res in result_vector}
text_filenames = {res["filename"] for res in result_text}

vect_filenames - text_filenames

{'02-vector-search/lessons/08-pgvector.md'}

In [50]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [54]:
## Q6

query_q6 = "How do I give the model access to tools?"
query_vect_q6 = embedder.encode(query_q6)

result_text_q6 = index.search(query_q6, num_results=5)
result_vector_q6 = vindex.search(query_vect_q6, num_results=5)

results_hybrid = rrf([result_vector_q6, result_text_q6])
results_hybrid[0]

{'start': 4000,
 'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function ca